# Principal Component Analysis (PCA) with WDI

This notebook is part of the English edition of the Big Data Analysis course materials.

This notebook demonstrates how PCA can be used to build a simplified composite development index from WDI indicators.

In [ ]:
from pathlib import Path

def find_project_root() -> Path:
    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "README.md").exists() and (candidate / "data").exists():
            return candidate
    return current

PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data"
ASSETS_DIR = PROJECT_ROOT / "assets"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PROJECT_ROOT


In [ ]:
MODULE_OUTPUT_DIR = OUTPUT_DIR / "05_pca"
MODULE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODULE_OUTPUT_DIR

# The Human Development Index (HDI) was introduced by the United Nations Development Programme in 1990.

## 1) HDI combines indicators related to longevity, education, and standard of living.
## 2) The official HDI is a composite index, but alternative composite measures can also be constructed.

![](../../assets/pca/2021-22_UN_Human_Development_Report.png)

# Q1: Can alternative methods be used to compute a composite index? That is, reducing three-dimensional data to one dimension? In other words: dimensionality reduction.
## Dimensionality reduction is a common data preprocessing technique in data analysis and machine learning. Its primary goal is to map high-dimensional data into a low-dimensional space while preserving the original data's information, thereby reducing complexity and improving model performance and interpretability.
### 1: Linear dimensionality reduction methods include:
### Principal Component Analysis (PCA): Uses linear transformation to project high-dimensional data into a low-dimensional space while retaining features with the highest variance. This is the most widely used dimensionality reduction method.
### Linear Discriminant Analysis (LDA): Aims to find projection directions that maximize between-class variance and minimize within-class variance, achieving dimensionality reduction. It is primarily used in supervised learning scenarios.

### 2: Nonlinear dimensionality reduction methods include:
### Kernel Principal Component Analysis (KPCA): An improvement over linear PCA using kernel methods. It applies a kernel function to nonlinearly transform samples and performs PCA in the transformed space, equivalent to nonlinear PCA in the original space.
### Locally Linear Embedding (LLE): A nonlinear dimensionality reduction algorithm that effectively preserves local structural information in high-dimensional data. It is well-suited for visualizing high-dimensional data.

# In Python, scikit-learn provides ready-to-use implementations of dimensionality-reduction algorithms.

## https://scikit-learn.org/stable/

# Q2:How to Create a Simplified Human Development Index Using PCA and World Bank WDI Data

## Step1: Read the data

In [ ]:
import numpy as np
import pandas as pd

### Load the lightweight WDI course subset.
df = pd.read_csv(DATA_DIR / "wdi" / "WDI_course_subset.csv")

### Select data for the United States, Singapore, China, Germany, Finland, Sweden, Denmark, Norway, South Korea, and Japan.
### Tip: data for high-income countries are often more complete.
data = df[df["Country Name"].isin([
    "United States",
    "Singapore",
    "China",
    "Germany",
    "Finland",
    "Sweden",
    "Denmark",
    "Norway",
    "Korea, Rep.",
    "Japan",
])]

### Convert the data to panel format.
data_pd = data.drop(columns="Indicator Code").melt(
    id_vars=["Country Name", "Country Code", "Indicator Name"],
    var_name="Year",
).pivot_table(
    values="value",
    index=["Country Name", "Country Code", "Year"],
    columns="Indicator Name",
).reset_index().rename_axis("", axis=1)

### Convert Year to integer.
data_pd["Year"] = data_pd["Year"].astype(str).astype(int)
data_pd.head()

## Step 2: Ensure that there is no missing data. If there are missing values, delete the missing values or interpolate the data.

In [ ]:
### Check the number of missing values in each column.
isna_data = data_pd.isna().sum().sort_values(ascending=True)
isna_data.to_csv(MODULE_OUTPUT_DIR / "isna_data.csv", index=True)
isna_data

In [ ]:
### Selection of variables for dimensionality reduction
### Here we use GDP per capita, life expectancy, and education expenditure as candidate ingredients of a simplified development index.
### You may either drop missing values or fill them with interpolation. PCA will fail if missing values remain.

# Example without interpolation:
# pca_data = data_pd.set_index(["Country Code", "Country Name", "Year"])[
#     [
#         "Life expectancy at birth, total (years)",
#         "Adjusted savings: education expenditure (% of GNI)",
#         "Government expenditure on education, total (% of GDP)",
#         "GDP per capita (constant 2015 US$)",
#         "GDP growth (annual %)",
#     ]
# ].query("Year < 2021 and Year > 2005").dropna()

# pca_data

In [ ]:
### The following code interpolates missing values within each country.
### Export intermediate results if you want to inspect them carefully.
df_missing = data_pd.set_index(["Country Code", "Country Name", "Year"])[
    [
        "Life expectancy at birth, total (years)",
        "Adjusted savings: education expenditure (% of GNI)",
        "Government expenditure on education, total (% of GDP)",
        "GDP per capita (constant 2015 US$)",
        "GDP growth (annual %)",
    ]
].query("Year < 2021 and Year > 2005")

def interpolate_group(group):
    if group.isna().any().any():
        group = group.interpolate(method="linear", limit_direction="both")
    return group

pca_data = df_missing.groupby("Country Name", group_keys=False).apply(interpolate_group)
pca_data

## Step3: Data standardization

In [ ]:
### Standardize the variables with `StandardScaler`.
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
scaled_data = scaler.fit_transform(pca_data)
scaled_data

## KMO and Barlett's Spherical Test determine whether a sample is suitable for principal component analysis.KMO tests for correlation and partial correlation between variables, and Barlett's Spherical Test is used to determine whether a correlation array is a unit array.

In [ ]:
%pip install -q factor_analyzer

## KMO Tests

In [ ]:
from factor_analyzer.factor_analyzer import calculate_kmo
kmo_all, kmo_model=calculate_kmo(scaled_data)
kmo_model

## Barlett's Spherical Test

In [ ]:
from factor_analyzer.factor_analyzer import calculate_bartlett_sphericity
chi_square_value,p_value=calculate_bartlett_sphericity(scaled_data)
chi_square_value, p_value

## A KMO statistic higher than 0.7 and a p-value of less than 0.05 for Barlett's test of sphericity were required.

## Step 4: Perform PCA on the data

![](../../assets/pca/pca_dimensionality_reduction_diagram.png)

In [ ]:
## PCA
### Call PCA from `sklearn`.
from sklearn.decomposition import PCA

pca = PCA(n_components=0.80)  ## Retain enough components to explain 80% of the variance.

### `fit_transform` reduces high-dimensional data to lower dimensions.
X_pca = pca.fit_transform(scaled_data)
X_pca

## Eigenvalues
### 1) Eigenvalues indicate the importance of each principal component in a principal component analysis, and are the magnitude of the total variance of the original variable explained by each principal component.
### 2) The first principal component has the largest eigenvalue, which explains the largest portion of the total variance of the original variable.
## 3) As the eigenvalues of the subsequent principal components become smaller, the variance of the original variable explained by them decreases.
### 4) The larger the eigenvalue, the more important the corresponding principal component is.

In [ ]:
eigenvalues = pca.explained_variance_
eigenvalues

## The variance contribution ratio
### 1) is a measure of how important each principal component is in explaining the variance of the original data.
### 2) Variance contribution ratio = eigenvalue of a principal component / sum of eigenvalues of all principal components.
### 3) The larger the eigenvalues of a principal component, the higher its contribution to variance.

In [ ]:
variance_contribution_rates = pca.explained_variance_ratio_
variance_contribution_rates

## Cumulative variance contribution ratio. The cumulative variance contribution rate is the percentage of the total variance of the original variable that can be explained by the selection of the first k principal components.

In [ ]:
cumulative_variance_contribution_rates = [sum(variance_contribution_rates[:i+1]) for i in range(len(variance_contribution_rates))]
cumulative_variance_contribution_rates

In [ ]:
%pip install -q openpyxl

In [ ]:
# Merge the eigenvalues, variance contribution rates, and cumulative variance contribution rates into one table.
pca_summary = pd.DataFrame({
    "Component": [f"PC{i+1}" for i in range(len(eigenvalues))],
    "Eigenvalue": eigenvalues,
    "Variance contribution rate": variance_contribution_rates,
    "Cumulative variance contribution rate": cumulative_variance_contribution_rates,
})

pca_summary.to_excel(MODULE_OUTPUT_DIR / "PCA_summary.xlsx", index=False)
pca_summary

## Step 5: Calculate the comprehensive index based on the decomposed principal components and merge it into the original data.

In [ ]:
# Convert the PCA result into a DataFrame and add column names.
pca_columns = [f"PC{i+1}" for i in range(X_pca.shape[1])]
df_pca = pd.DataFrame(X_pca, columns=pca_columns)

# Compute the composite PCA index.
df_pca["pca_index"] = (-X_pca * pca.explained_variance_ratio_).sum(axis=1)

# Merge the index back to country and year information.
df_pca = pd.concat(
    [df_pca["pca_index"], pca_data.reset_index()[["Country Code", "Country Name", "Year"]]],
    axis=1,
    join="inner",
)
df_pca

## Step 6: Sort according to the comprehensive index

In [ ]:
df_pca

In [ ]:
# Rank countries by the PCA-based index within each year.
df_pca["Rank"] = df_pca.groupby(["Year"])["pca_index"].rank(method="first", ascending=False)
df_pca.to_csv(MODULE_OUTPUT_DIR / "df_pca_ranked.csv", index=False)
df_pca

In [ ]:
%pip install -q seaborn

## Step 7: Present the calculated comprehensive indicators in a graph

In [ ]:
%pip install -q colorcet

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import colorcet as cc

### Specify a colormap with 10 distinct colors.
palette = sns.color_palette(cc.glasbey, n_colors=10)

### Draw the line plot.
sns.lineplot(data=df_pca, x="Year", y="pca_index", hue="Country Name", palette=palette)

### Set the x-axis ticks.
plt.xticks(np.arange(2005, 2021, 5))

plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.savefig(MODULE_OUTPUT_DIR / "pca_index_lineplot.png", bbox_inches="tight")

In [ ]:
country_names = pca_data.reset_index()["Country Name"]
pca_result = pd.DataFrame(X_pca, columns=["PC1", "PC2"])
pca_result["Country Name"] = country_names

# Create a scatter plot.
plt.figure(figsize=(8, 6))
sns.scatterplot(
    x=-pca_result["PC1"],
    y=-pca_result["PC2"],
    hue=pca_result["Country Name"],
    palette=palette,
)
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.title("PCA Visualization")
plt.legend()
plt.savefig(MODULE_OUTPUT_DIR / "pca_scatter.png", bbox_inches="tight")

In [ ]:
### Draw a scree plot.
plt.figure(figsize=(8, 6))
plt.plot(range(1, len(eigenvalues) + 1), eigenvalues, marker="o")
plt.xlabel("Principal component")
plt.ylabel("Eigenvalue")
plt.title("Scree plot")

# Add a horizontal line at y=1 to mark the turning point.
plt.axhline(y=1, color="r", linestyle="--")
plt.savefig(MODULE_OUTPUT_DIR / "scree_plot.png", bbox_inches="tight")